# Deploy Wan2.1-T2V-1.3B-Diffusers on SageMaker using the vLLM-Omni DLC

This notebook deploys the [Wan-AI/Wan2.1-T2V-1.3B-Diffusers](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers) text-to-video model to an Amazon SageMaker AI asynchronous endpoint using the vLLM-Omni Deep Learning Container (DLC).

It creates the SageMaker model, endpoint configuration, and endpoint, then submits inline JSON requests. Generated MP4 files are written asynchronously to Amazon S3.

> **Recommended environment:** Amazon SageMaker Studio.

> **Cost notice:** The endpoint incurs charges while it is provisioned. Run the cleanup section when finished.

## 1. Environment preparation

### Prerequisites

Before running the notebook, confirm that:

- Your AWS Region supports the selected instance type and has sufficient service quota.
- The S3 bucket exists in the same Region as the endpoint.
- Your identity can create SageMaker resources and pass the execution role.
- The execution role can read and write the configured S3 locations.
- The endpoint has network access to download the model from Hugging Face.

Update the bucket placeholder below to the name of a bucket owned by your AWS account.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

### AWS clients and notebook settings

The following cell discovers the current AWS Region, creates SageMaker control-plane and runtime clients, and sets the S3 bucket used for asynchronous inference results.

In [ ]:
import time
import re
import json
import boto3
from IPython.display import clear_output

region = boto3.Session().region_name

sm = boto3.client("sagemaker")
sm_runtime = boto3.client("sagemaker-runtime")

bucket = "<YOUR_VIDEO_OUTPUT_BUCKET>"

### SageMaker helper functions

These helpers derive the SageMaker execution role and wait for endpoint creation without requiring the SageMaker Python SDK.

In [ ]:
def get_sagemaker_role():
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    return re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", arn)


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    progress = ""
    while True:
        status = describe_fn(**{name_key: name})[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )

### SageMaker execution role

When running in SageMaker Studio, the helper function derives the execution role from the current assumed-role identity.

When running elsewhere, set `role` explicitly to a SageMaker execution-role ARN. The caller must also have permission to pass this role to SageMaker.

In [ ]:
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

## 2. Configuration

The deployment uses the vLLM-Omni SageMaker DLC and one GPU. No `ModelDataUrl` is supplied when the SageMaker model is created. Instead, the container downloads the model identified by `SM_VLLM_MODEL` during startup.

The following timeouts serve different purposes:

- `ContainerStartupHealthCheckTimeoutInSeconds` controls how long SageMaker waits for the container to become healthy.
- `VLLM_OMNI_VIDEO_SYNC_TIMEOUT`, when configured, controls how long the container permits an individual generation request to run.

Model size, frame count, resolution, and inference-step count can substantially affect startup time, GPU memory use, and generation latency.

See the [vLLM-Omni SageMaker video deployment example](https://github.com/aws/deep-learning-containers/blob/main/examples/vllm-omni/sagemaker/deploy_video_sync.py) for additional context.

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:omni-sagemaker-cuda-v1.6"

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

instance = {"type": "ml.g6e.2xlarge", "num_gpu": 1}

model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 600
variant_name = "v1"

## 3. Model deployment

Model deployment on Amazon SageMaker AI consists of three steps:

1. [Create a model](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/create_model.html) that defines the serving container and its environment. In this notebook, the container downloads the model from Hugging Face during startup.
2. [Create an endpoint configuration](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/create_endpoint_config.html) that defines the asynchronous inference settings, instance type and count, concurrency, and S3 output locations.
3. [Create an endpoint](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/sagemaker/client/create_endpoint.html) and wait for it to become available.

### Authentication and container environment

> **Security:** Never hard-code or commit a Hugging Face access token in a notebook. If model access requires authentication, retrieve the token from a secure secret source and add it to the container environment only when needed.

The container environment identifies the model to load and configures tensor parallelism. Tensor parallelism should match the number of GPUs assigned to each endpoint instance.

To override the per-request processing timeout, add `"VLLM_OMNI_VIDEO_SYNC_TIMEOUT": "1800"` to the environment dictionary.

In [ ]:
common_env = {
    "HF_TOKEN": "<YOUR_TOKEN>",
    #"VLLM_OMNI_VIDEO_SYNC_TIMEOUT": "1800",  # set to maximum number of seconds your request can take
}
vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
}
env = common_env | vllm_env

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

### Why use asynchronous inference?

Video generation can be long-running. SageMaker asynchronous inference accepts the request, queues it, and writes the eventual result to S3.

`MaxConcurrentInvocationsPerInstance` is set to `1`, so each instance processes one video-generation request at a time. This reduces GPU contention but means that additional requests may wait in the queue.

Successful outputs are written to `S3OutputPath`, while invocation failures are written to `S3FailurePath`. SNS notifications are optional and can be enabled when clients need completion events.

To enable notifications, add a `NotificationConfig` entry containing the success and error SNS topic ARNs to `OutputConfig`.

Although the container route is `/v1/videos/sync`, the overall SageMaker invocation remains asynchronous: SageMaker manages the queue while the container processes each request synchronously.

In [ ]:
async_config = {
    "ClientConfig": {
        "MaxConcurrentInvocationsPerInstance": 1
    },
    "OutputConfig": {
        "S3OutputPath": f"s3://{bucket}/async/out",
        "S3FailurePath": f"s3://{bucket}/async/err"
    }
}

sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
    AsyncInferenceConfig=async_config,
)

sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)

wait_for_endpoint(endpoint_name)

## 4. Inference examples

Each example submits an inline JSON request to the asynchronous endpoint; it does not wait for the MP4 file to be produced. Retain the returned inference ID and S3 locations so the request can be tracked.

The three examples are submitted consecutively. Because concurrency is limited to one request per instance, they will normally be processed sequentially.

### Request parameters

| Parameter | Purpose |
|---|---|
| `prompt` | Describes the desired video |
| `num_frames` | Controls video length and strongly affects generation time |
| `num_inference_steps` | Trades generation speed for output quality |
| `size` | Controls output resolution and GPU-memory requirements |
| `seed` | Helps reproduce a generation under equivalent runtime conditions |

Start with smaller frame counts and resolutions when validating a new deployment.

For more information, see the [asynchronous inference inline payload example](https://github.com/aws-samples/sagemaker-genai-hosting-examples/tree/main/03-features/async-inference-inline-payload).

In [ ]:
payload = {
    "prompt": "a dog running on a beach",
    "num_frames": "161",
    "num_inference_steps": "50",
    "size": "640x480",
    "seed": "1"
}

res = sm_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Filename="test_video1.mp4",
    Body=json.dumps(payload).encode("utf-8"),
    CustomAttributes="route=/v1/videos/sync",
)
print(json.dumps(res, indent=2))

In [ ]:
payload = {
    "prompt": "A curious raccoon standing and looking directly at the camera near a garbage bin",
    "num_frames": "161",
    "num_inference_steps": "50",
    "size": "640x480",
    "seed": "1"
}

res = sm_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Filename="test_video2.mp4",
    Body=json.dumps(payload).encode("utf-8"),
    CustomAttributes="route=/v1/videos/sync",
)
print(json.dumps(res, indent=2))

In [ ]:
payload = {
    "prompt": "A traditional Christmas dinner table with candles and presents",
    "num_frames": "161",
    "num_inference_steps": "50",
    "size": "640x480",
    "seed": "1"
}

res = sm_runtime.invoke_endpoint_async(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Filename="test_video3.mp4",
    Body=json.dumps(payload).encode("utf-8"),
    CustomAttributes="route=/v1/videos/sync",
)
print(json.dumps(res, indent=2))

### Retrieving results

A successful submission response identifies where the generated video will be written. Wait for the S3 output object or consume the configured SNS notification before attempting to download it.

If generation fails:

- Inspect the configured S3 failure location.
- Review the endpoint's CloudWatch logs.
- Check for model-download, GPU-memory, timeout, and S3-permission errors.
- Verify that the requested resolution and frame count fit the selected instance.

## 5. Cleanup

Run this section even if an earlier inference request fails. Endpoint charges continue until the endpoint has been deleted.

Cleanup does not delete generated videos, failure records, SNS topics, or other objects stored in S3.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_model(ModelName=model_name)